# Welcome to DSL Course FPGA Lab -

Welcome to the DSL Course! This notebook platform is used to **develop**, **simulate** and **implement** **Xilinx FPGA** designs directly from **Jupyter**, without opening the Vivado GUI manually each time. 
Before we start, let’s check that the **Vivado toolchain** is correctly installed and available on this server:

In [1]:
# Environment Check-Up
! vivado -version
%load_ext vivado_magic

vivado v2025.1 (64-bit)
Tool Version Limit: 2025.05
SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.


## Step 1: Configure Your Project

Here we define the basic project settings used by the Vivado magic extension:  
- `project_name`: logical name of your design project  
- `device_part`: target FPGA device (CMOD A7 in this example)  
- `design_top`: top-level RTL module  
- `sim_top`: top-level testbench module  

We also run `%vivado prj_clean` once to remove any old build directory so we can start from a fresh project state.


In [11]:
device_part  = "xc7a35tcpg236-1" # Do not touch
project_name = "demo"
design_top   = "top_module"
sim_top      = "tb_top_module"
%vivado prj_clean

[INFO]: Removing build directory: /home/user/ug2600/Dsl_notebook/build_demo


## Step 2: Create the Top-Level Design (with Lint Check)

In this cell we define the top-level RTL module `top_module` using the `%%vivado_design` magic.  
The code is saved as `top_module.sv` under the project’s `src/hdl` folder, and the tool automatically runs a Verilator lint check to catch basic syntax and style issues before we go into synthesis.

In [12]:
%%vivado_design top_module
module top_module(output pio47, input pio48);
    assign pio47 = ~pio48;
endmodule

[INFO]: Creating design file: /home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv
[INFO]: Running Verilator lint:
        verilator --sv --lint-only -Wno-TIMESCALEMOD -Wno-MULTITOP /home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv

[INFO]: Verilator lint completed with no errors (exit code 0).
[INFO]: Vivado project '/home/user/ug2600/Dsl_notebook/build_demo/demo/demo.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


## Step 3: Write a Simple Testbench (with Lint Check)

Here we create a basic testbench `tb_top_module` using the `%%vivado_sim` magic.  
This cell saves the testbench as `tb_top_module.sv` under `src/tb`, instantiates the `top_module` module, and applies a few stimulus pulses to `pio48`. Just like for the design, Verilator lint is run automatically on both the design and testbench to catch common issues before simulation or synthesis.

In [13]:
%%vivado_sim tb_top_module
`timescale 1ns/1ps
module tb_top_module;
    logic a, y;
    top_module dut(.pio48(a), .pio47(y));

    // Simple "monitor" using always + $display (Verilator-friendly)
    initial begin
        $display(" time(ns)   a   y");
    end

    always @(a, y) begin
        $display("%8t   %0b   %0b", $time, a, y);
    end

    initial begin
        a = 0;
        #5 a = 1;
        #5 a = 0;
        #10 $finish;
    end
endmodule

[INFO]: Creating testbench file: /home/user/ug2600/Dsl_notebook/build_demo/src/tb/tb_top_module.sv
[INFO]: Running Vivado xvlog lint:
        xvlog -sv /home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv /home/user/ug2600/Dsl_notebook/build_demo/src/tb/tb_top_module.sv
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv" into library work
INFO: [VRFC 10-311] analyzing module top_module
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2600/Dsl_notebook/build_demo/src/tb/tb_top_module.sv" into library work
INFO: [VRFC 10-311] analyzing module tb_top_module

[INFO]: Vivado xvlog lint completed with no errors (exit code 0).
[INFO]: Vivado project '/home/user/ug2600/Dsl_notebook/build_demo/demo/demo.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


## Step 4: Add Board Constraints (XDC)

This cell uses the `%%vivado_xdc` magic to register a constraint file `cmod_pins.xdc` for the Digilent Cmod A7 board.  
The XDC describes how FPGA pins are mapped to external signals (clock, LEDs, buttons, PMOD, GPIO, etc.).  

For this simple demo we only enable the GPIO mappings for `pio47` and `pio48`, which connect directly to the `top_module` module ports. The file is saved under `src/constraints` and will be picked up by Vivado when we create and implement the project.

In [14]:
%%vivado_xdc cmod_pins
set_property -dict { PACKAGE_PIN U8    IOSTANDARD LVCMOS33 } [get_ports { pio47 }]; #IO_L14P_T2_SRCC_34 Sch=pio[47]
set_property -dict { PACKAGE_PIN V8    IOSTANDARD LVCMOS33 } [get_ports { pio48 }]; #IO_L14N_T2_SRCC_34 Sch=pio[48]

[INFO]: Creating XDC file: /home/user/ug2600/Dsl_notebook/build_demo/src/constraints/cmod_pins.xdc
[INFO]: Vivado project '/home/user/ug2600/Dsl_notebook/build_demo/demo/demo.xpr' not found; skipping auto-add.
[INFO]: The file is saved under build/src and will be picked up when you run '%vivado prj_create'.


## Step 5: Create the Vivado Project

Now we call `%vivado prj_create` to generate a full Vivado project under `build_<project_name>`.  

This command:
- Scans the `src/hdl`, `src/tb`, and `src/constraints` folders created in the previous steps  
- Adds the design, testbench, and XDC files to the project  
- Sets `Top` as the synthesis top module and `tbTop` as the simulation top  

After this, the project is ready for synthesis, implementation, and simulation.

In [15]:
%vivado prj_create

[INFO]: Running: vivado -mode batch -source /tmp/tmpmmkyxdap/cmd.tcl -tclargs demo xc7a35tcpg236-1 top_module tb_top_module /home/user/ug2600/Dsl_notebook/build_demo/src/hdl /home/user/ug2600/Dsl_notebook/build_demo/src/tb /home/user/ug2600/Dsl_notebook/build_demo/src/constraints /home/user/ug2600/Dsl_notebook/build_demo

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Wed Dec 10 12:39:29 2025
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

[INFO]: Project name  : demo
[INFO]: Device part   : xc7a35tcpg236-1
[INFO]: Design top    : top_module
[INFO]: Sim top       : tb_top_module
[INFO]: Project dir   : /home/user/ug2600/Dsl_notebook/build_demo/demo
[INFO]: Output dir    : /home/user/ug2600/Dsl_notebook/build_dem

## Step 6: Run the Behavior Simulation;

Finally, we launch the Vivado Simulation "xsim", to run the simulation. You can view the text meassage in Cell output. If you want to view the waveform, there are two methods:
1. Web-Based

    a. Download the VCD file from "build_\<Your Project Name\>/output/*.vcd"
   
    b. Open the [Surfer](https://app.surfer-project.org); 

    c. Open the VCD file;

2. GUI-Based

    a. Download the VCD file from "build_\<Your Project Name\>/output/*.vcd"

    b. You can also Surfer Application via the [link](https://surfer-project.org);

    c. Open the VCD file via local appliction Surfer;



In [16]:
%vivado sim_rtl

[INFO]: Running XSIM compile (xvlog):
        xvlog -sv /home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv /home/user/ug2600/Dsl_notebook/build_demo/src/tb/tb_top_module.sv
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2600/Dsl_notebook/build_demo/src/hdl/top_module.sv" into library work
INFO: [VRFC 10-311] analyzing module top_module
INFO: [VRFC 10-2263] Analyzing SystemVerilog file "/home/user/ug2600/Dsl_notebook/build_demo/src/tb/tb_top_module.sv" into library work
INFO: [VRFC 10-311] analyzing module tb_top_module

[INFO]: Running XSIM elaborate (xelab):
        xelab tb_top_module -s tb_top_module -debug typical --timescale 1ns/1ps
Vivado Simulator v2025.1.0
Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.
Running: /home/user/eda/AMD/2025.1/Vivado/bin/unwrapped/lnx64.o/xelab tb_top_module -s tb_top_module -debug typical --timescale 1ns/1ps 
Multi-threading is on. Using 118 sla

## Step 7: Synthesis and Implementation

In this step we run the full hardware flow from the notebook:

- `%vivado syn` runs synthesis for the `Top` module on the selected FPGA.
- `%vivado imp` runs implementation and generates the final bitstream (saved under `build_<project_name>/output/`).

After these commands finish successfully, you will have a `.bit` file ready to be programmed onto the Cmod A7 board.

In [17]:
%vivado syn
%vivado imp

[INFO]: Running: vivado -mode batch -source /tmp/tmpuu6qe4ah/cmd.tcl -tclargs demo /home/user/ug2600/Dsl_notebook/build_demo top_module xc7a35tcpg236-1

****** Vivado v2025.1 (64-bit)
  **** SW Build 6140274 on Wed May 21 22:58:25 MDT 2025
  **** IP Build 6138677 on Thu May 22 03:10:11 MDT 2025
  **** SharedData Build 6139179 on Tue May 20 17:58:58 MDT 2025
  **** Start of session at: Wed Dec 10 13:05:57 2025
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

INFO: [filemgmt 56-3] Default IP Output Path : Could not find the directory '/home/user/ug2600/Dsl_notebook/build_demo/demo/demo.gen/sources_1'.
Scanning sources...
Finished scanning sources
[INFO]: Starting synthesis...
INFO: [Project 1-1160] Copying file /home/user/ug2600/Dsl_notebook/build_demo/demo/demo.runs/synth_1/top_module.dcp to /home/user/ug2600/Dsl_notebook/build_demo/demo/demo.srcs/utils_1/imports/synth_1 and adding it to utils file